In [1]:
%pip -q install python-docx

In [2]:
"""Лабораторная работа №2: MLP в PyTorch на Fashion-MNIST.

Запуск: python lab2_fashion_mnist.py --fio "Фамилия И.О." --group "ИСТ-XX"
Файлы появятся в lab2_results, включая Word-отчёт с ФАКТИЧЕСКИМИ результатами.
Для полного дополнительного эксперимента: --bonus-dropout.
Для короткой проверки без датасета: --smoke-test (НЕ для сдачи!).
"""
import argparse
import csv
import json
import os
import random
import time
from copy import deepcopy
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)

CLASSES = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
           'Sandal','Shirt','Sneaker','Bag','Ankle boot']
MEAN, STD = 0.2860, 0.3530
SEED = 42
OPT_SPECS = {
    'SGD': {'method':'SGD','lr':0.01,'momentum':0.0},
    'SGD + momentum': {'method':'SGD','lr':0.01,'momentum':0.9},
    'Adam': {'method':'Adam','lr':0.001},
    'AdamW': {'method':'AdamW','lr':0.001,'weight_decay':0.0001},
}


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_sizes=(256,128), num_classes=10,
                 dropout_rate=0.0, use_batchnorm=False):
        super().__init__()
        layers=[]; prev=input_size
        for h in hidden_sizes:
            layers.append(nn.Linear(prev,h))
            if use_batchnorm: layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout_rate: layers.append(nn.Dropout(dropout_rate))
            prev=h
        layers.append(nn.Linear(prev,num_classes))
        self.net=nn.Sequential(*layers)
    def forward(self,x):
        return self.net(x.reshape(x.size(0),-1))  # RAW logits for CE


class MLPWithSoftmax(MLP):
    def forward(self,x):
        return torch.softmax(super().forward(x),dim=1)


def parameters_breakdown(model):
    return [(name, int(p.numel())) for name,p in model.named_parameters() if p.requires_grad]


def get_data(data_dir, smoke=False):
    if smoke:
        # Только для внутренней проверки исполняемости; в отчёт не попадает.
        g=torch.Generator().manual_seed(SEED)
        xs=torch.randn((500,1,28,28),generator=g)
        ys=torch.randint(0,10,(500,),generator=g)
        tr=TensorDataset(xs,ys); te=TensorDataset(xs[:100],ys[:100])
    else:
        import torchvision
        from torchvision import transforms
        transform=transforms.Compose([
            transforms.ToTensor(), transforms.Normalize((MEAN,), (STD,))
        ])
        tr=torchvision.datasets.FashionMNIST(root=str(data_dir),train=True,download=True,transform=transform)
        te=torchvision.datasets.FashionMNIST(root=str(data_dir),train=False,download=True,transform=transform)
    nt=int(len(tr)*0.90); nv=len(tr)-nt
    train_ds,val_ds=random_split(tr,[nt,nv],generator=torch.Generator().manual_seed(SEED))
    return tr,te,train_ds,val_ds


def loaders(train_ds,val_ds,test_ds,batch_size=64,seed=SEED):
    # Перезапускаемый генератор: один и тот же порядок батчей для честного сравнения.
    train=DataLoader(train_ds,batch_size=batch_size,shuffle=True,num_workers=0,
                     generator=torch.Generator().manual_seed(seed))
    val=DataLoader(val_ds,batch_size=256,shuffle=False,num_workers=0)
    test=DataLoader(test_ds,batch_size=256,shuffle=False,num_workers=0)
    return train,val,test


def loss_for(out,y,mode):
    if mode=='CE': return nn.functional.cross_entropy(out,y)
    oh=nn.functional.one_hot(y,num_classes=10).to(dtype=out.dtype)
    return nn.functional.mse_loss(out,oh)


def train_epoch(model,loader,optimizer,device,mode='CE'):
    model.train(); running=0.0; hits=0; n=0
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        optimizer.zero_grad(set_to_none=True)
        out=model(x)
        loss=loss_for(out,y,mode)
        loss.backward(); optimizer.step()
        b=len(y); running+=float(loss.item())*b; hits+=int((out.argmax(1)==y).sum()); n+=b
    return running/n,hits/n


@torch.no_grad()
def evaluate(model,loader,device,mode='CE',include_probs=False):
    model.eval(); running=0.0; hits=0; n=0; preds=[]; truths=[]; scores=[]
    for x,y in loader:
        x,y=x.to(device),y.to(device)
        out=model(x)
        loss=loss_for(out,y,mode)
        p=out.argmax(1); b=len(y)
        running+=float(loss.item())*b; hits+=int((p==y).sum()); n+=b
        preds.append(p.cpu().numpy()); truths.append(y.cpu().numpy())
        if include_probs:
            prob=out if mode=='MSE' else torch.softmax(out,dim=1)
            scores.append(prob.cpu().numpy())
    return {'loss':running/n,'acc':hits/n,'y_pred':np.concatenate(preds),
            'y_true':np.concatenate(truths),
            'probs':np.concatenate(scores) if include_probs else None}


def optimizer_for(model,name):
    kw=dict(OPT_SPECS[name]); method=kw.pop('method')
    return getattr(torch.optim,method)(model.parameters(),**kw)


def fit(model,train_loader,val_loader,optimizer,device,epochs=30,patience=7,
        mode='CE',csv_path=None,verbose=True):
    hist={'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[], 'time_sec':[]}
    best_val_loss=float('inf'); best_state=deepcopy(model.state_dict()); best_epoch=0
    stagnant=0
    for epoch in range(1,epochs+1):
        t0=time.perf_counter()
        tr_loss,tr_acc=train_epoch(model,train_loader,optimizer,device,mode)
        v=evaluate(model,val_loader,device,mode)
        elapsed=time.perf_counter()-t0
        for k,value in {'train_loss':tr_loss,'train_acc':tr_acc,
                        'val_loss':v['loss'],'val_acc':v['acc'],'time_sec':elapsed}.items():
            hist[k].append(float(value))
        if v['loss'] < best_val_loss-1e-8:
            best_val_loss=float(v['loss']); best_state=deepcopy(model.state_dict())
            best_epoch=epoch; stagnant=0
        else: stagnant+=1
        if verbose: print(f'  epoch {epoch:2d}/{epochs}: train acc={tr_acc:.4f}, '
                          f'val acc={v["acc"]:.4f}, val loss={v["loss"]:.4f}',flush=True)
        if stagnant>=patience:
            if verbose: print(f'  early stopping at epoch {epoch}; best loss at {best_epoch}',flush=True)
            break
    model.load_state_dict(best_state)
    hist['best_epoch_by_val_loss']=int(best_epoch)
    if csv_path: write_log(hist,csv_path)
    return hist


def write_log(hist,path):
    with open(path,'w',newline='',encoding='utf-8-sig') as f:
        writer=csv.writer(f); writer.writerow(['epoch','train_loss','train_acc','val_loss','val_acc','time_sec'])
        for idx in range(len(hist['train_loss'])):
            writer.writerow([idx+1]+[f'{hist[key][idx]:.6f}' for key in
                                     ['train_loss','train_acc','val_loss','val_acc','time_sec']])


def plot_architecture(path):
    fig,ax=plt.subplots(figsize=(10,1.75))
    ax.set(xlim=(0,10),ylim=(0,1));ax.axis('off')
    labs=['28×28\nFlatten','784\nfeatures','Linear\n256','ReLU\n(+BN/Drop)','Linear\n128','ReLU\n(+BN/Drop)','Linear\n10 logits']
    xs=np.linspace(.7,9.3,7)
    for j,(x,lab) in enumerate(zip(xs,labs)):
        ax.text(x,.52,lab,ha='center',va='center',fontsize=10,
                bbox=dict(boxstyle='round,pad=.45',facecolor='#e9eef6',edgecolor='#577ca4'))
        if j<6: ax.annotate('',xy=(x+(xs[1]-xs[0])*.49,.52),xytext=(x+(xs[1]-xs[0])*.34,.52),
                            arrowprops=dict(arrowstyle='->',lw=1.4,color='#577ca4'))
    fig.tight_layout(pad=.3);fig.savefig(path,dpi=180,bbox_inches='tight');plt.close(fig)


def plot_examples(dataset,path):
    g=np.random.default_rng(SEED); inds=g.choice(len(dataset),size=9,replace=False)
    fig,axs=plt.subplots(3,3,figsize=(7,7))
    for ax,idx in zip(axs.flat,inds):
        img,y=dataset[int(idx)]
        ax.imshow((img.squeeze().numpy()*STD+MEAN).clip(0,1),cmap='gray',vmin=0,vmax=1)
        ax.set_title(CLASSES[int(y)],fontsize=10); ax.axis('off')
    fig.tight_layout();fig.savefig(path,dpi=155,bbox_inches='tight');plt.close(fig)


def plot_hist(hist,title,path):
    fig,axs=plt.subplots(1,2,figsize=(10,3.6))
    for key in ('train_loss','val_loss'): axs[0].plot(range(1,len(hist[key])+1),hist[key],label=key)
    for key in ('train_acc','val_acc'): axs[1].plot(range(1,len(hist[key])+1),hist[key],label=key)
    axs[0].set_title('Loss'); axs[1].set_title('Accuracy')
    for ax in axs: ax.set_xlabel('Epoch');ax.grid(alpha=.2);ax.legend(fontsize=8)
    fig.suptitle(title);fig.tight_layout();fig.savefig(path,dpi=165,bbox_inches='tight');plt.close(fig)


def plot_compare_loss(ce,mse,path):
    fig,axs=plt.subplots(1,2,figsize=(10,3.6))
    for tag,h in [('Cross-Entropy',ce),('MSE + Softmax',mse)]:
        axs[0].plot(range(1,len(h['train_loss'])+1),h['train_loss'],label=tag)
        axs[1].plot(range(1,len(h['val_acc'])+1),h['val_acc'],label=tag)
    axs[0].set_title('Train loss (different scales!)');axs[1].set_title('Val accuracy')
    for ax in axs: ax.set_xlabel('Epoch');ax.grid(alpha=.2);ax.legend(fontsize=8)
    fig.tight_layout();fig.savefig(path,dpi=165,bbox_inches='tight');plt.close(fig)


def plot_optimizers(op_results,path):
    fig,ax=plt.subplots(figsize=(9,4))
    for name,res in op_results.items():
        a=res['history']['val_acc'];ax.plot(range(1,len(a)+1),a,label=name)
    ax.set_xlabel('Epoch');ax.set_ylabel('Validation accuracy');ax.grid(alpha=.2);ax.legend()
    fig.tight_layout();fig.savefig(path,dpi=165,bbox_inches='tight');plt.close(fig)


def plot_regularization(reg_results,path):
    fig,axs=plt.subplots(1,2,figsize=(10,3.6))
    for name,res in reg_results.items():
        h=res['history']
        for ax,metric in zip(axs,['train_loss','val_loss']):
            ax.plot(range(1,len(h[metric])+1),h[metric],label=name)
    axs[0].set_title('Train loss');axs[1].set_title('Validation loss')
    for ax in axs:ax.set_xlabel('Epoch');ax.grid(alpha=.2);ax.legend(fontsize=8)
    fig.tight_layout();fig.savefig(path,dpi=165,bbox_inches='tight');plt.close(fig)


def plot_cm(cm,path):
    fig,ax=plt.subplots(figsize=(9,7))
    ConfusionMatrixDisplay(cm,display_labels=CLASSES).plot(ax=ax,cmap='Blues',xticks_rotation=35,values_format='d',colorbar=False)
    ax.set_title('Confusion matrix (test)');fig.tight_layout();fig.savefig(path,dpi=165,bbox_inches='tight');plt.close(fig)


def top_pairs(cm,num=2):
    pairs=[(int(cm[a,b]+cm[b,a]),a,b) for a in range(10) for b in range(a+1,10)]
    return sorted(pairs,reverse=True)[:num]


def plot_errors(ds,ev,pairs,path):
    fig,axs=plt.subplots(2,5,figsize=(13,5.5))
    truths=ev['y_true']; preds=ev['y_pred']; probs=ev['probs']
    for row,(_,a,b) in enumerate(pairs):
        ix=np.flatnonzero(((truths==a)&(preds==b))|((truths==b)&(preds==a)))[:5]
        for col,ax in enumerate(axs[row]):
            if col<len(ix):
                idx=int(ix[col]);im,y=ds[idx]
                ax.imshow((im.squeeze().numpy()*STD+MEAN).clip(0,1),cmap='gray',vmin=0,vmax=1)
                ax.set_title(f'{CLASSES[int(y)]} → {CLASSES[int(preds[idx])]}\np={probs[idx,int(preds[idx])]:.2f}',fontsize=8)
            else: ax.text(.5,.5,'No example',ha='center',va='center')
            ax.axis('off')
    fig.tight_layout();fig.savefig(path,dpi=165,bbox_inches='tight');plt.close(fig)


def plot_bonus(bonus,path):
    fig,ax=plt.subplots(figsize=(7,3.5))
    xs=sorted(bonus);ys=[bonus[x]['test_acc'] for x in xs]
    ax.plot(xs,ys,marker='o');ax.set_xlabel('Dropout rate');ax.set_ylabel('Test accuracy')
    ax.grid(alpha=.2);fig.tight_layout();fig.savefig(path,dpi=165,bbox_inches='tight');plt.close(fig)


def train_run(name,train_ds,val_ds,test_ds,device,out,optimizer_name,
              model_kw=None,mode='CE',epochs=30,patience=7,log=True):
    model_kw=model_kw or {};set_seed(SEED)
    tr,val,te=loaders(train_ds,val_ds,test_ds)
    model=(MLPWithSoftmax if mode=='MSE' else MLP)(**model_kw).to(device)
    opt=optimizer_for(model,optimizer_name)
    print(f'=== {name} | {optimizer_name} | {mode} | {model_kw} ===',flush=True)
    t=time.perf_counter()
    h=fit(model,tr,val,opt,device,epochs=epochs,patience=patience,mode=mode,
          csv_path=out/f'{name}.csv' if log else None,verbose=False)
    elapsed=time.perf_counter()-t
    check=evaluate(model,val,device,mode)
    test=evaluate(model,te,device,mode)
    print(f'  DONE: epochs={len(h["val_acc"])} best-val-acc={max(h["val_acc"]):.4f} '
          f'test-acc={test["acc"]:.4f} time={elapsed:.1f}s',flush=True)
    return {'history':h,'val_acc':check['acc'],'best_val_acc':max(h['val_acc']),
            'test_acc':test['acc'],'time_sec':elapsed,'model_kwargs':model_kw}


def run_work(out,data_dir,device,bonus=False,quick=False,smoke=False):
    all_train,test_ds,train_ds,val_ds=get_data(data_dir,smoke)
    plot_examples(all_train,out/'figure_1_samples.png')
    plot_architecture(out/'figure_2_architecture.png')
    epochs=3 if quick else 30; nloss=3 if quick else 15; nbest=3 if quick else 40
    patience=7
    base_kw={'dropout_rate':0.0,'use_batchnorm':False}
    base=train_run('baseline_sgd',train_ds,val_ds,test_ds,device,out,'SGD',base_kw,epochs=epochs,patience=patience)
    plot_hist(base['history'],'SGD baseline',out/'figure_3_baseline.png')

    ce=train_run('loss_ce',train_ds,val_ds,test_ds,device,out,'SGD',base_kw,epochs=nloss,patience=nloss+1)
    mse=train_run('loss_mse',train_ds,val_ds,test_ds,device,out,'SGD',base_kw,mode='MSE',epochs=nloss,patience=nloss+1)
    plot_compare_loss(ce['history'],mse['history'],out/'figure_4_loss_comparison.png')

    opt_results={'SGD':base}
    for name in ['SGD + momentum','Adam','AdamW']:
        opt_results[name]=train_run('opt_'+name.replace(' ','_').replace('+','plus'),
                                   train_ds,val_ds,test_ds,device,out,name,base_kw,epochs=epochs,patience=patience)
    plot_optimizers(opt_results,out/'figure_5_optimizers.png')
    best_opt=max(opt_results,key=lambda x:opt_results[x]['best_val_acc'])
    print('Best optimizer selected by VAL accuracy:',best_opt)

    reg_kws={'Baseline':base_kw,
             'Dropout 0.3':{'dropout_rate':.3,'use_batchnorm':False},
             'Dropout + BatchNorm':{'dropout_rate':.3,'use_batchnorm':True}}
    reg_results={}
    for name,kw in reg_kws.items():
        reg_results[name]=train_run('reg_'+name.replace(' ','_').replace('+','plus'),
                                    train_ds,val_ds,test_ds,device,out,best_opt,kw,epochs=epochs,patience=patience)
    plot_regularization(reg_results,out/'figure_6_regularization.png')
    best_reg=max(reg_results,key=lambda x:reg_results[x]['best_val_acc'])
    print('Best regularization selected by VAL accuracy:',best_reg)

    set_seed(SEED)
    tr,val,te=loaders(train_ds,val_ds,test_ds)
    model=MLP(**reg_kws[best_reg]).to(device)
    optimizer=optimizer_for(model,best_opt)
    hist=fit(model,tr,val,optimizer,device,epochs=nbest,patience=patience,csv_path=out/'best_training_log.csv',verbose=False)
    evaluation=evaluate(model,te,device,include_probs=True)
    plot_hist(hist,'Best MLP',out/'figure_7_best_history.png')
    cm=confusion_matrix(evaluation['y_true'],evaluation['y_pred'],labels=list(range(10)))
    plot_cm(cm,out/'figure_8_confusion_matrix.png')
    pairs=top_pairs(cm)
    plot_errors(test_ds,evaluation,pairs,out/'figure_9_error_examples.png')
    torch.save({'model_state':model.state_dict(),'model_kwargs':reg_kws[best_reg],
                'optimizer':best_opt,'class_names':CLASSES},out/'best_model.pth')
    metrics={'test_loss':evaluation['loss'], 'test_accuracy':evaluation['acc'],
             'precision_macro':precision_score(evaluation['y_true'],evaluation['y_pred'],average='macro',zero_division=0),
             'recall_macro':recall_score(evaluation['y_true'],evaluation['y_pred'],average='macro',zero_division=0),
             'f1_macro':f1_score(evaluation['y_true'],evaluation['y_pred'],average='macro',zero_division=0)}
    bonus_results={}
    if bonus:
        for d in [0.1,0.2,0.3,0.5]:
            bonus_results[d]=train_run(f'bonus_dropout_{int(d*100)}',train_ds,val_ds,test_ds,
                                       device,out,best_opt,{'dropout_rate':d,'use_batchnorm':False},
                                       epochs=epochs,patience=patience)
        plot_bonus(bonus_results,out/'figure_10_dropout_bonus.png')
    data={'dataset':'Fashion-MNIST' if not smoke else 'SYNTHETIC_SMOKE_TEST',
          'dataset_counts':{'all_train':len(all_train),'train':len(train_ds),'val':len(val_ds),'test':len(test_ds)},
          'torch':torch.__version__,'device':str(device),
          'base':base,'loss':{'CE':ce,'MSE':mse},'optimizers':opt_results,'regularization':reg_results,
          'best_optimizer':best_opt,'best_regularization':best_reg,'best_history':hist,
          'best_metrics':metrics,'confusion_matrix':cm.tolist(),
          'problem_pairs':[{'count':count,'a':CLASSES[a],'b':CLASSES[b]} for count,a,b in pairs],
          'bonus_dropout':{str(k):v for k,v in bonus_results.items()},
          'parameters_basic':parameters_breakdown(MLP()),
          'parameters_best':sum(p.numel() for p in model.parameters() if p.requires_grad)}
    with open(out/'results.json','w',encoding='utf-8') as f: json.dump(data,f,ensure_ascii=False,indent=2)
    return data


def make_report(data,out,fio='________________',group='________________'):
    from docx import Document
    from docx.shared import Cm,Pt,RGBColor
    from docx.enum.text import WD_ALIGN_PARAGRAPH,WD_LINE_SPACING
    from docx.enum.table import WD_TABLE_ALIGNMENT,WD_CELL_VERTICAL_ALIGNMENT
    from docx.enum.style import WD_STYLE_TYPE
    from docx.oxml import OxmlElement
    from docx.oxml.ns import qn

    doc=Document(); sec=doc.sections[0]
    sec.top_margin=sec.bottom_margin=Cm(2)
    sec.left_margin=Cm(3);sec.right_margin=Cm(1.5)
    sec.header_distance=sec.footer_distance=Cm(1.1)
    sec.different_first_page_header_footer=True
    styles=doc.styles
    normal=styles['Normal'];normal.font.name='Times New Roman';normal.font.size=Pt(14)
    normal.paragraph_format.line_spacing=1.5
    normal.paragraph_format.first_line_indent=Cm(1.25)
    normal.paragraph_format.space_after=Pt(0)
    for name in ['Heading 1','Heading 2']:
        sty=styles[name];sty.font.name='Times New Roman';sty.font.size=Pt(14);sty.font.bold=True;sty.font.color.rgb=RGBColor(0,0,0)
        sty.paragraph_format.first_line_indent=Cm(0)
        sty.paragraph_format.space_before=Pt(10);sty.paragraph_format.space_after=Pt(6)
        sty.paragraph_format.keep_with_next=True
    def para(txt='',bold=False,align=WD_ALIGN_PARAGRAPH.JUSTIFY,indent=True):
        p=doc.add_paragraph();p.alignment=align
        if not indent:p.paragraph_format.first_line_indent=Cm(0)
        if bold:p.add_run(txt).bold=True
        else:p.add_run(txt)
        return p
    def heading(txt): doc.add_heading(txt,level=1)
    def img(file,number,caption,width=13.2):
        fp=out/file
        if fp.exists():
            p=doc.add_paragraph();p.alignment=WD_ALIGN_PARAGRAPH.CENTER
            p.paragraph_format.first_line_indent=Cm(0);p.paragraph_format.space_before=Pt(4)
            p.add_run().add_picture(str(fp),width=Cm(width))
            p2=para(f'Рисунок {number} – {caption}',align=WD_ALIGN_PARAGRAPH.CENTER,indent=False)
            p2.paragraph_format.space_after=Pt(6)
    def tbl(number,caption,headers,rows):
        p=para(f'Таблица {number} – {caption}',align=WD_ALIGN_PARAGRAPH.LEFT,indent=False)
        p.paragraph_format.space_before=Pt(6)
        table=doc.add_table(rows=1, cols=len(headers));table.style='Table Grid';table.alignment=WD_TABLE_ALIGNMENT.CENTER
        for c,h in zip(table.rows[0].cells,headers):c.text=str(h)
        for row in rows:
            cells=table.add_row().cells
            for c,v in zip(cells,row):c.text=str(v)
        for ri,row in enumerate(table.rows):
            for cell in row.cells:
                cell.vertical_alignment=WD_CELL_VERTICAL_ALIGNMENT.CENTER
                for p in cell.paragraphs:
                    p.paragraph_format.first_line_indent=Cm(0)
                    p.paragraph_format.line_spacing=1.0
                    p.paragraph_format.space_after=Pt(0)
                    for r in p.runs:
                        r.font.name='Times New Roman';r.font.size=Pt(10.5)
                        if ri==0:r.bold=True
        return table
    f=lambda n:f'{n:.4f}'
    if data['dataset']=='SYNTHETIC_SMOKE_TEST':
        raise ValueError('Нельзя создавать сдаваемый отчёт по искусственным данным!')
    # Title
    for line in ['МИНИСТЕРСТВО НАУКИ И ВЫСШЕГО ОБРАЗОВАНИЯ РОССИЙСКОЙ ФЕДЕРАЦИИ',
                 'ПОВОЛЖСКИЙ ГОСУДАРСТВЕННЫЙ ТЕХНОЛОГИЧЕСКИЙ УНИВЕРСИТЕТ']:
        para(line,align=WD_ALIGN_PARAGRAPH.CENTER,indent=False)
    for _ in range(5):para('',indent=False)
    para('ОТЧЁТ ПО ЛАБОРАТОРНОЙ РАБОТЕ № 2',bold=True,align=WD_ALIGN_PARAGRAPH.CENTER,indent=False)
    para('по дисциплине «Машинное обучение и анализ данных»',align=WD_ALIGN_PARAGRAPH.CENTER,indent=False)
    para('«Обучение MLP в PyTorch — классификация изображений»',align=WD_ALIGN_PARAGRAPH.CENTER,indent=False)
    for _ in range(5):para('',indent=False)
    para(f'Выполнила: {fio}',align=WD_ALIGN_PARAGRAPH.RIGHT,indent=False)
    para(f'Группа: {group}',align=WD_ALIGN_PARAGRAPH.RIGHT,indent=False)
    para('Проверил: _______________________',align=WD_ALIGN_PARAGRAPH.RIGHT,indent=False)
    for _ in range(2):para('',indent=False)
    para('Йошкар-Ола — 2026',align=WD_ALIGN_PARAGRAPH.CENTER,indent=False)
    doc.add_page_break()
    # Footer page field
    footer=sec.footer.paragraphs[0];footer.alignment=WD_ALIGN_PARAGRAPH.CENTER
    footer.paragraph_format.first_line_indent=Cm(0)
    field=OxmlElement('w:fldSimple');field.set(qn('w:instr'),'PAGE');footer._p.append(field)

    heading('Цель работы')
    para('Изучить обучение многослойного перцептрона средствами PyTorch, работу автоматического дифференцирования, сравнить функции потерь, оптимизаторы и способы регуляризации на задаче классификации изображений одежды.')
    heading('1. Исходные данные и подготовка')
    para('Использован датасет Fashion-MNIST (Xiao и др., 2017): 60 000 обучающих и 10 000 тестовых изображений размером 28×28 пикселей, 10 классов одежды. Обучающая часть разделена случайным воспроизводимым образом на 54 000 объектов обучения и 6 000 объектов валидации (seed = 42). Тестовые данные не участвуют в обучении и выборе гиперпараметров.')
    para('Изображения преобразованы в тензоры, нормализованы с mean = 0,2860 и std = 0,3530. Для обучения DataLoader формирует батчи по 64 изображения с перемешиванием; валидация и тестирование выполняются без перемешивания.')
    img('figure_1_samples.png',1,'Девять случайных изображений Fashion-MNIST',width=9.4)
    para('Внешне сходные категории, например Shirt и T-shirt/top, Pullover и Coat, могут путаться из-за небольшого разрешения и сходных силуэтов.')
    heading('2. Архитектура многослойного перцептрона')
    para('Реализована полносвязная сеть 784 → 256 → 128 → 10: входное изображение разворачивается в вектор длины 784; два скрытых слоя имеют активацию ReLU; последний линейный слой выдаёт 10 логитов. При необходимости после линейных слоёв добавляется BatchNorm1d, после ReLU — Dropout. Softmax перед CrossEntropyLoss не применяется.')
    img('figure_2_architecture.png',2,'Блок-схема прямого прохода полносвязной сети',width=13.2)
    tbl(1,'Количество обучаемых параметров по слоям', ['Слой','Параметры'],
        [['Linear 784→256','200 960'],['Linear 256→128','32 896'],['Linear 128→10','1 290'],['Всего (без BatchNorm)','235 146'],['Дополнительные параметры двух BatchNorm','768']])
    para('В первой лабораторной вход имел значительно меньше признаков; здесь первый слой содержит 784×256 весов, поэтому число параметров существенно выше. Прямой проход: Flatten → Linear → (BatchNorm) → ReLU → (Dropout) → Linear → (BatchNorm) → ReLU → (Dropout) → Linear → logits. На обратном проходе autograd вычисляет градиенты, которые оптимизатор использует для обновления весов.')
    heading('3. Базовая модель и сравнение функций потерь')
    b=data['base']; bh=b['history']
    para(f'Базовая сеть обучалась с SGD (lr = 0,01, momentum = 0) не более 30 эпох, patience = 7. Достигнута лучшая validation accuracy {f(b["best_val_acc"])}; accuracy на тесте — {f(b["test_acc"])}. Количество завершённых эпох: {len(bh["val_acc"])}.')
    img('figure_3_baseline.png',3,'Динамика loss и accuracy базовой MLP')
    gap=bh['train_acc'][-1]-bh['val_acc'][-1]
    para(f'На последней эпохе разница train/validation accuracy составила {gap:.3f}. '
         + ('Это свидетельствует о наличии разрыва обобщения, который необходимо контролировать регуляризацией.' if gap>0.03 else 'Значительного разрыва между обучением и валидацией по порогу 0,03 не наблюдалось.')
         + ' Окончательная оценка зависит также от динамики loss и момента достижения лучших значений валидации.')
    l=data['loss']; ce=l['CE']['history']; mse=l['MSE']['history']
    tbl(2,'Сравнение Cross-Entropy и MSE за 15 эпох', ['Функция потерь','Train loss (15)','Val acc (15)','Best val acc'],
        [['CE',f(ce['train_loss'][-1]),f(ce['val_acc'][-1]),f(max(ce['val_acc']))],
         ['MSE + Softmax',f(mse['train_loss'][-1]),f(mse['val_acc'][-1]),f(max(mse['val_acc']))]])
    img('figure_4_loss_comparison.png',4,'Сравнение динамики функций потерь и validation accuracy')
    lead='Cross-Entropy' if np.mean(ce['val_acc'][:min(5,len(ce['val_acc']))]) >= np.mean(mse['val_acc'][:min(5,len(mse['val_acc']))]) else 'MSE'
    final='Cross-Entropy' if ce['val_acc'][-1]>=mse['val_acc'][-1] else 'MSE'
    para(f'В первые пять эпох по средней validation accuracy лидировала {lead}; к 15-й эпохе лучший результат показала {final}. Значения MSE и CE не сравниваются напрямую из-за различных масштабов функций потерь; сравнивают скорость изменения метрик и итоговую точность. При MSE градиент по логитам содержит якобиан Softmax и может ослабевать при насыщении, тогда как CE даёт устойчивую форму градиента p − y. CrossEntropyLoss принимает логиты и внутри вычисляет log-softmax устойчивым способом; добавление Softmax перед CE меняет оптимизируемую функцию и обычно ухудшает градиентный сигнал.')
    heading('4. Сравнение оптимизаторов')
    o=data['optimizers']
    tbl(3,'Результаты оптимизаторов при одинаковой архитектуре',
        ['Оптимизатор','Best val acc','Test acc','Время, с'],
        [[k,f(v['best_val_acc']),f(v['test_acc']),f'{v["time_sec"]:.1f}'] for k,v in o.items()])
    img('figure_5_optimizers.png',5,'Validation accuracy для четырёх оптимизаторов')
    para(f'По максимальной validation accuracy выбран {data["best_optimizer"]}. В первых эпохах Adam/AdamW часто ускоряют сходимость за счёт адаптивного масштабирования шага для каждого веса. SGD без момента может обучаться медленнее, а SGD с momentum учитывает накопленное направление градиента и способен догонять адаптивные методы при длительном обучении. AdamW отделяет weight decay от адаптивного шага. Время зависит от вычислительного устройства; для чистого сравнения запусков использовались одинаковые данные, архитектура и seed. Выбор сделан по валидационной, а не тестовой точности.')
    heading('5. Влияние Dropout и BatchNorm')
    reg=data['regularization'];tbl(4,'Эксперименты с регуляризацией',
        ['Конфигурация','Best val acc','Test acc','Время, с'],
        [[k,f(v['best_val_acc']),f(v['test_acc']),f'{v["time_sec"]:.1f}'] for k,v in reg.items()])
    img('figure_6_regularization.png',6,'Train и validation loss для трёх конфигураций')
    para(f'По validation accuracy выбрана конфигурация «{data["best_regularization"]}». Dropout (p = 0,3) на обучении случайно обнуляет часть активаций и уменьшает зависимость модели от отдельных нейронов. BatchNorm нормализует промежуточные активации в пределах батча; в режиме eval вместо статистик текущего батча применяются накопленные значения. Чтобы оценить пользу регуляризации, сравнивают train/val gap и test accuracy для трёх одинаково обученных вариантов, а не только минимальный train loss.')
    heading('6. Итоговая модель и анализ ошибок')
    m=data['best_metrics']
    para(f'Лучшая конфигурация заново обучена не более 40 эпох с ранней остановкой. Её параметры: оптимизатор — {data["best_optimizer"]}; регуляризация — {data["best_regularization"]}; число обучаемых параметров — {data["parameters_best"]:,}.')
    tbl(5,'Метрики итоговой MLP на тестовой выборке', ['Метрика','Значение'],
        [['Accuracy',f(m['test_accuracy'])],['Precision (macro)',f(m['precision_macro'])],
         ['Recall (macro)',f(m['recall_macro'])],['F1 (macro)',f(m['f1_macro'])],['Cross-Entropy loss',f(m['test_loss'])]])
    img('figure_7_best_history.png',7,'Кривые loss и accuracy итоговой модели')
    img('figure_8_confusion_matrix.png',8,'Матрица ошибок на тестовой выборке',width=10.5)
    pp=data['problem_pairs'];para('Наиболее часто путаемые пары (сумма ошибок в двух направлениях): '+ '; '.join(f'{x["a"]} ↔ {x["b"]} — {x["count"]}' for x in pp) + '.')
    img('figure_9_error_examples.png',9,'Примеры ошибок по двум наиболее проблемным парам')
    para('Сходство силуэтов и отсутствие пространственной структуры после Flatten затрудняют различение похожих классов. Для изображений более подходящей альтернативой является сверточная сеть (CNN), сохраняющая локальные пространственные признаки.')
    if data.get('bonus_dropout'):
        heading('7. Дополнительный эксперимент: величина Dropout')
        img('figure_10_dropout_bonus.png',10,'Test accuracy для различных коэффициентов Dropout')
        para('Изменялось значение dropout_rate (0,1; 0,2; 0,3; 0,5) при прочих одинаковых условиях. Эти результаты дополнительного исследования не использовались для выбора итоговой модели: её гиперпараметры отбирались на валидации.')
    heading('Выводы')
    para('1. Реализован MLP на PyTorch с автоматическим вычислением градиентов, режимами train/eval и ранней остановкой по валидационной потере.')
    para(f'2. В серии сравнений оптимизаторов наибольшую validation accuracy показал {data["best_optimizer"]}. Для итоговой модели по валидации выбрана регуляризация «{data["best_regularization"]}».')
    para(f'3. Итоговая модель достигла test accuracy {f(m["test_accuracy"])} и macro-F1 {f(m["f1_macro"])}. Анализ матрицы ошибок выявил сложности разграничения внешне похожих категорий.')
    para('4. Для улучшения качества перспективны свёрточные модели (CNN), расширение поиска гиперпараметров и оценка устойчивости результатов на нескольких случайных разбиениях.')
    heading('Ответы на контрольные вопросы')
    qa=[
        ('1. NumPy и PyTorch, роль autograd.','В NumPy все производные для backpropagation вычислялись вручную. PyTorch строит вычислительный граф и автоматически рассчитывает градиенты через loss.backward(), позволяя менять архитектуру без ручного пересчёта производных.'),
        ('2. Назначение optimizer.zero_grad().','Обнуляет градиенты предыдущего шага. Без этого PyTorch накапливает их в полях .grad, что приводит к непреднамеренному суммированию градиентов разных батчей.'),
        ('3. Почему CrossEntropyLoss принимает логиты.','Функция сама выполняет log-softmax и отрицательное логарифмическое правдоподобие устойчивым способом. Softmax до неё приводит к повторному преобразованию вероятностей как логитов и изменяет оптимизируемую функцию.'),
        ('4. Adam и SGD с моментом.','SGD с моментом сглаживает обновления, накапливая направление градиента; Adam дополнительно нормирует обновления по оценке второго момента отдельно для каждого параметра. Adam обычно быстрее в начале, SGD с подобранным расписанием иногда лучше обобщает в конце.'),
        ('5. BatchNorm, train и eval.','BatchNorm использует статистики текущего батча в train и накопленные running mean/variance в eval. Если забыть model.eval(), результат зависит от состава батча и искажается, особенно на маленьких батчах.'),
        ('6. Dropout на обучении и тесте.','Во время обучения случайно выключает часть нейронов, снижая переобучение. На тесте выключен, иначе результаты становятся случайными и возможна потеря точности.'),
        ('7. Что будет при увеличении глубины до пяти слоёв по 1024 нейрона.','Растут число параметров, вычисления и риск переобучения. Проверять улучшение нужно на одной и той же validation выборке при одинаковом протоколе обучения; итоговый test использовать один раз.'),
        ('8. Почему CIFAR-10 сложнее для MLP.','Цветные изображения содержат больше деталей, объектов и пространственных отношений. MLP при Flatten теряет локальную структуру, поэтому CNN обычно лучше: свёртки извлекают локальные признаки и переиспользуют фильтры.'),
        ('9. Различие model.train() и model.eval().','train() включает Dropout и использование статистик текущего батча в BatchNorm; eval() отключает Dropout и включает накопленные статистики BatchNorm. eval() сам по себе не отключает вычисление градиентов — дополнительно используют torch.no_grad().'),
        ('10. Подбор learning rate для Adam.','Сначала проверяют диапазон около 10⁻⁴–10⁻³ и сравнивают кривые валидации, при необходимости применяют scheduler и раннюю остановку. Шаг 0,1 обычно слишком велик для Adam и может привести к нестабильному обучению.')
    ]
    for q,a in qa:
        para(q,bold=True);para(a)
    heading('Использованные источники')
    para('1. Umbratick. Лабораторная работа № 2: «Обучение MLP в PyTorch — классификация изображений». URL: https://github.com/umbratick/ml-da/blob/main/sem2/lab2.md (дата обращения: 11.11.2026).')
    para('2. Xiao H., Rasul K., Vollgraf R. Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms. 2017. arXiv:1708.07747.')
    para('3. PyTorch Documentation. URL: https://pytorch.org/docs/stable/ (дата обращения: 11.11.2026).')
    target=out/'Lab2_Report.docx'
    doc.save(target)
    return target


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--fio',default='________________________')
    parser.add_argument('--group',default='________________')
    parser.add_argument('--out',default='lab2_results')
    parser.add_argument('--data-dir',default='data')
    parser.add_argument('--bonus-dropout',action='store_true')
    parser.add_argument('--quick',action='store_true',help='Быстрый тест, не для сдачи')
    parser.add_argument('--smoke-test',action='store_true',help='Искусственные данные, не для сдачи')
    args=parser.parse_args()
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    set_seed();device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type=='cpu': torch.set_num_threads(min(os.cpu_count() or 2,4))
    print('PyTorch:',torch.__version__,'device:',device,flush=True)
    result=run_work(out,Path(args.data_dir),device,bonus=args.bonus_dropout,
                    quick=args.quick,smoke=args.smoke_test)
    if not args.smoke_test and not args.quick:
        path=make_report(result,out,args.fio,args.group)
        print('Готовый отчёт:',path.resolve(),flush=True)
    else: print('ПРОВЕРОЧНЫЙ ЗАПУСК: окончательный отчёт не создан.',flush=True)
    print('Все файлы в',out.resolve(),flush=True)



In [3]:
# ИЗМЕНИТЕ СВОИ ДАННЫЕ ПЕРЕД ЗАПУСКОМ
FIO = "Хазгалеева О.С."
GROUP = "ИСТ-45"
BONUS_DROPOUT = False  # True: дополнительные 4 эксперимента (+10%)

from pathlib import Path
import zipfile
from google.colab import files

OUT = Path('/content/lab2_results')
OUT.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Используемое устройство:', device)
if device.type == 'cpu':
    torch.set_num_threads(4)

# Эксперименты на РЕАЛЬНОМ Fashion-MNIST (скачивается при первом запуске)
results = run_work(OUT, Path('/content/data'), device,
                   bonus=BONUS_DROPOUT, quick=False, smoke=False)
report_path = make_report(results, OUT, fio=FIO, group=GROUP)
print('ГОТОВО! Итоговый отчёт:', report_path)

# Дополнительно упакуем графики, логи, метрики и отчёт в единый архив.
zip_path = Path('/content/Lab2_Files.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)
print('Архив результатов:', zip_path)
files.download(str(report_path))
# При необходимости скачайте архив отдельным действием:
# files.download(str(zip_path))

Используемое устройство: cuda


100%|██████████| 26.4M/26.4M [00:02<00:00, 11.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 174kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.30MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 20.2MB/s]


=== baseline_sgd | SGD | CE | {'dropout_rate': 0.0, 'use_batchnorm': False} ===
  DONE: epochs=30 best-val-acc=0.8897 test-acc=0.8789 time=392.4s
=== loss_ce | SGD | CE | {'dropout_rate': 0.0, 'use_batchnorm': False} ===
  DONE: epochs=15 best-val-acc=0.8808 test-acc=0.8741 time=190.7s
=== loss_mse | SGD | MSE | {'dropout_rate': 0.0, 'use_batchnorm': False} ===
  DONE: epochs=15 best-val-acc=0.7618 test-acc=0.7653 time=196.3s
=== opt_SGD_plus_momentum | SGD + momentum | CE | {'dropout_rate': 0.0, 'use_batchnorm': False} ===
  DONE: epochs=15 best-val-acc=0.8942 test-acc=0.8862 time=201.4s
=== opt_Adam | Adam | CE | {'dropout_rate': 0.0, 'use_batchnorm': False} ===
  DONE: epochs=15 best-val-acc=0.8930 test-acc=0.8845 time=210.4s
=== opt_AdamW | AdamW | CE | {'dropout_rate': 0.0, 'use_batchnorm': False} ===
  DONE: epochs=15 best-val-acc=0.8950 test-acc=0.8862 time=199.7s
Best optimizer selected by VAL accuracy: AdamW
=== reg_Baseline | AdamW | CE | {'dropout_rate': 0.0, 'use_batchnorm'

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from pathlib import Path
from IPython.display import display, Image

graphs = sorted(Path("/content/lab2_results").glob("figure_*.png"))

if graphs:
    for graph in graphs:
        print(graph.name)
        display(Image(filename=str(graph), width=800))
else:
    print("Графики не найдены в текущей среде.")

Графики не найдены в текущей среде.
